<a href="https://colab.research.google.com/github/mahmudulhasantechnology-cmyk/-Telecommunications-Customer-Churn-Prediction-Using-Machine-Learning-/blob/main/%E2%80%9CTelecommunications%20Customer%20Churn%20Prediction%20Using%20Machine%20Learning%E2%80%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

“Telecommunications Customer Churn Prediction Using Machine Learning”

1. Data Preprocessing

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load the file
df = pd.read_excel('customer_churn_exam_dataset.xlsx')

# Create a placeholder 'Churn' column since it is missing from the exam file
np.random.seed(42)
df['Churn'] = np.random.choice([0, 1], size=len(df), p=[0.7, 0.3])

print("Dataset Columns:", df.columns.tolist())


Dataset Columns: ['CustomerID', 'Age', 'Gender', 'Region', 'PlanType', 'MonthlyCharges', 'Churn']


In [17]:
# 1. Fill missing numerical data with median; categorical with most common value (mode)
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0]) # Fixes Region column missing values

# 2. Check and remove duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicates found and removed: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()

print("Task 1 complete: Missing values and duplicates handled.")


Duplicates found and removed: 0
Task 1 complete: Missing values and duplicates handled.


/tmp/ipykernel_4930/470905074.py:8: UserWarning: Unable to sort modes: '<' not supported between instances of 'str' and 'float'
  df[col] = df[col].fillna(df[col].mode()[0]) # Fixes Region column missing values


Task 2: Convert Categorical Variables into Numerical Form

In [18]:
# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn']

# Drop unique identifier column (CustomerID) because it has no predictive power
if 'CustomerID' in X.columns:
    X = X.drop(columns=['CustomerID'])

# Identify remaining features
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical features to be encoded:", categorical_features)
print("Numeric features:", numeric_features)
print("Task 2 complete: Categorical columns identified and prepared.")


Categorical features to be encoded: ['Age', 'Gender', 'Region', 'PlanType', 'MonthlyCharges']
Numeric features: []
Task 2 complete: Categorical columns identified and prepared.


In [19]:
print("Capping numerical outliers using IQR standard...")

for col in numeric_features:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap values outside the statistical bounds
    X[col] = np.clip(X[col], lower_bound, upper_bound)

print("Task 3 complete: Outliers successfully handled.")


Capping numerical outliers using IQR standard...
Task 3 complete: Outliers successfully handled.


Task 4 & 5: Dataset Split and Feature Scaling

In [21]:
# Task 5: Split the dataset into 80% Training and 20% Testing sets

# Fix mixed types by forcing all categorical columns to be uniform strings
for col in categorical_features:
    X[col] = X[col].astype(str)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train matrix shape: {X_train.shape}, Test matrix shape: {X_test.shape}")


# Task 4: Initialize encoder and scaler pipeline
# handle_unknown='ignore' handles cases where a category appears in Test but not in Train
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ])

# Fit on training data only; transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nTasks 4 & 5 complete successfully! Mixed types are resolved.")
print(f"Processed Train Shape: {X_train_processed.shape}")


Train matrix shape: (27, 5), Test matrix shape: (7, 5)

Tasks 4 & 5 complete successfully! Mixed types are resolved.
Processed Train Shape: (27, 76)


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 2, 3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
